## Improve the model by grouping PUMAs into NTA

Geographic areas are naturally hierarchical—blocks sit inside neighborhoods, and neighborhoods sit inside cities.

In our case, PUMAs nest inside NTAs. In this notebook, we group PUMAs under their corresponding NTAs and model them hierarchically. 
This lets the model share information across nearby neighborhoods, so individual PUMAs borrow strength from their broader 
NTA when data is sparse. The result is more stable estimates and more realistic patterns of noise complaints from neighborhood to neighborhood.


This process of making the model Hherarchical models is called partial pooling because they let neighborhoods borrow strength from each other instead of being treated as completely independent.

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API",
    category=UserWarning,
)

In [2]:
import pymc as pm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import arviz as az
import geopandas as gpd
from keplergl import KeplerGl
from helpers import ( prep_the_data, 
                      export_puma_kepler, 
                      make_daily_table_for_model_with_nta,
                      load_idata,

                      export_idata,
                      compare_models_loo_waic,
                      loo_diagnostics,
                        kepler_typical_week_from_dow_complaint
                    )




/Users/mozilla/Library/Caches/pypoetry/virtualenvs/nyc-311-bayesian-noise-models-pweGKFeb-py3.12/lib/python3.12/site-packages/arviz/__init__.py:39: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


### Load + prepare data: 2021-2024 

In [3]:
df_puma = pd.read_parquet(
    "../data/processed/features/puma_noise_counts.parquet"
)


In [4]:
df_puma = prep_the_data(df_puma)

In [5]:
df_puma_2021__2024 = df_puma.loc[
    (df_puma["created_bucket"] >= "2021-01-01") &
    (df_puma["created_bucket"] <  "2024-12-31") 
].copy()


In [6]:
df_puma_2021__2024.head()

,puma,complaint_type,descriptor,location_type,created_bucket,time_of_day,complaint_count,nta,area_share_of_puma,nta_name,nta_puma,dow,month,is_weekend,month_year,descriptor_group,dow_complaint
0,4103,Noise,Noise,NR5,2022-06-09,day,1,MN0303,0.403905,East Village,East Village — 4103,Thursday,June,0,June__2022,Other,OTHER__Thursday
1,4103,Noise,"Noise, Barking Dog",NaN,2021-07-10,night,1,MN0303,0.403905,East Village,East Village — 4103,Saturday,July,1,July__2021,Animal,ANIMAL__Saturday
2,4103,Noise,"Noise, Barking Dog",NaN,2021-08-07,day,1,MN0303,0.403905,East Village,East Village — 4103,Saturday,August,1,August__2021,Animal,ANIMAL__Saturday
3,4103,Noise,"Noise, Barking Dog",NaN,2021-08-13,day,1,MN0303,0.403905,East Village,East Village — 4103,Friday,August,0,August__2021,Animal,ANIMAL__Friday
4,4103,Noise,"Noise, Barking Dog",NaN,2021-08-15,night,2,MN0303,0.403905,East Village,East Village — 4103,Sunday,August,1,August__2021,Animal,ANIMAL__Sunday


In [7]:
daily_df, coords = make_daily_table_for_model_with_nta(
    df_puma_2021__2024,
    complaint_col="descriptor_group",
)

This model recognizes that neighborhoods don’t exist in isolation. PUMAs are grouped within NTAs, 
and the model first learns shared weekday patterns at the NTA level before allowing individual 
PUMAs to deviate when the data supports it. 

In [8]:
# daily_df has: puma, nta_name, puma_idx, nta_idx, dow_idx, daily_count

# Build a unique mapping: puma_idx -> nta_idx
puma_nta_map = (
    daily_df[["puma_idx", "nta_idx"]]
    .drop_duplicates()
    .sort_values("puma_idx")
)

# sanity check: each puma maps to exactly 1 nta
assert puma_nta_map["puma_idx"].is_unique, "A PUMA maps to multiple NTAs (check your join)."

puma_to_nta_idx = puma_nta_map["nta_idx"].to_numpy()  # shape (n_puma,)


### Constructing the Group Model 

This is the actual data we observed.

In [9]:
y = daily_df["daily_count"].to_numpy(dtype=int)


This tells the model which PUMA each observation belongs to.

In [10]:
puma_idx_obs = daily_df["puma_idx"].to_numpy(dtype=int)

This tells the model what day of the week each observation occurred on.

In [11]:
dow_idx_obs  = daily_df["dow_idx"].to_numpy(dtype=int)

This is the moment where the hierarchy actually happens. Take the NTA’s weekday pattern, and then add 
this PUMA’s adjustment.”

```python
log_lambda = pm.Deterministic(
    "log_lambda",
    mu_nta[puma_to_nta_idx, :] + delta_puma,
)
```

A Deterministic variable is where everything comes together. It isn’t something the model is guessing 
on its own — it’s something the model calculates from what it has already learned.

```python
lam = pm.Deterministic("lam", pm.math.exp(log_lambda), dims=("puma", "dow"))
```

In this case:

    - mu_nta = the NTA’s weekday baseline

    - delta_puma = the PUMA’s adjustment

    - log_lambda = their combined effect
    
    - lam = the final expected number of complaints

So you can think of lam as:

"Given everything the model has learned, this is the expected number of complaints for each PUMA on each weekday."

In [12]:
with pm.Model(coords=coords) as m3_nta:

    # NTA-level weekday baseline on log scale
    mu_nta = pm.Normal("mu_nta", mu=0.0, sigma=1.5, dims=("nta", "dow"))

    # PUMA-level deviation around its NTA baseline
    sigma_puma = pm.Exponential("sigma_puma", 1.0)
    delta_puma = pm.Normal("delta_puma", mu=0.0, sigma=sigma_puma, dims=("puma", "dow"))

    # Build log-rate for every (puma, dow) using PUMA->NTA mapping (NOT per-observation indexing)
    log_lambda = pm.Deterministic(
        "log_lambda",
        mu_nta[puma_to_nta_idx, :] + delta_puma,
        dims=("puma", "dow"),
    )

    lam = pm.Deterministic("lam", pm.math.exp(log_lambda), dims=("puma", "dow"))

    # Likelihood: each observation picks its (puma, dow)
    y_obs = pm.Poisson("y_obs", mu=lam[puma_idx_obs, dow_idx_obs], observed=y)

    idata_puma_nta_pois = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=42,
        idata_kwargs={"log_likelihood": True},
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [mu_nta, sigma_puma, delta_puma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2206 seconds.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 1 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 2 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 3 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.


In [13]:
az.summary(idata_puma_nta_pois)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"mu_nta[Annadale-Huguenot-Prince's Bay-Woodrow, Monday]",0.400,1.340,-2.124,2.991,0.029,0.022,2110.0,2398.0,1.00
"mu_nta[Annadale-Huguenot-Prince's Bay-Woodrow, Tuesday]",0.362,1.322,-2.182,2.800,0.031,0.019,1872.0,2670.0,1.00
"mu_nta[Annadale-Huguenot-Prince's Bay-Woodrow, Wednesday]",0.478,1.319,-2.134,2.803,0.029,0.021,2130.0,2449.0,1.00
"mu_nta[Annadale-Huguenot-Prince's Bay-Woodrow, Thursday]",0.420,1.349,-2.143,2.915,0.032,0.020,1824.0,2562.0,1.00
"mu_nta[Annadale-Huguenot-Prince's Bay-Woodrow, Friday]",0.415,1.397,-2.047,3.102,0.033,0.020,1757.0,2679.0,1.00
...,...,...,...,...,...,...,...,...,...
"lam[4503, Wednesday]",8.140,0.455,7.254,8.979,0.008,0.011,3681.0,1709.0,1.00
"lam[4503, Thursday]",8.375,0.470,7.477,9.221,0.007,0.008,4030.0,2466.0,1.00
"lam[4503, Friday]",9.679,0.469,8.777,10.532,0.008,0.009,3521.0,2114.0,1.00
"lam[4503, Saturday]",21.230,0.715,19.956,22.584,0.011,0.013,4141.0,2713.0,1.01


###  compare both models
Now that we have both models we want to compare the previous flat model to the current heirarchical model

In [14]:
idata_puma_pois = load_idata("../data/processed/models/model1_puma_pois_idata.nc")

✅ Loaded idata <- ../data/processed/models/model1_puma_pois_idata.nc


This table is comparing two different models and asking a simple question:

Which model does a better job predicting new, unseen noise complaint data?

To answer that, we use two standard scoring methods (LOO and WAIC). You don’t need to understand the math 
behind them—just how to interpret the results.

The ELPD is the most important value in the table. Think of this as a prediction score.
    -  Higher (or less negative) = better predictions

In this cas, the Poisson + NTA pooling model performs better than the flat PUMA×weekday model.

In [15]:
loo_table, waic_table = compare_models_loo_waic(idata_puma_pois, idata_puma_nta_pois, m2_name="Poisson PUMA×DOW", m3_name="Poisson + NTA pooling")


/Users/mozilla/Library/Caches/pypoetry/virtualenvs/nyc-311-bayesian-noise-models-pweGKFeb-py3.12/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(
/Users/mozilla/Library/Caches/pypoetry/virtualenvs/nyc-311-bayesian-noise-models-pweGKFeb-py3.12/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to

,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
Poisson + NTA pooling,0,-163478.182174,5853.608413,0.000000,0.5,10512.063790,0.000000,True,log
Poisson PUMA×DOW,1,-163544.332609,5732.581140,66.150434,0.5,10634.647016,152.718234,True,log


,rank,elpd_waic,p_waic,elpd_diff,weight,se,dse,warning,scale
Poisson + NTA pooling,0,-167573.272167,9948.698405,0.000000,0.5,11734.773506,0.000000,True,log
Poisson PUMA×DOW,1,-167662.928358,9851.176889,89.656191,0.5,11863.348532,168.362358,True,log


These numbers are a health check on the model comparison, not a measure of model quality itself. The metric to focus 
on is pareto_k.

| `pareto_k` value | What it means                | Is this okay?        |
| ---------------- | ---------------------------- | -------------------- |
| **≤ 0.5**        | Observation behaves normally | ✅ Very good          |
| **0.5 – 0.7**    | Slightly influential         | ✅ Fine               |
| **> 0.7**        | Strong influence             | ⚠️ Worth checking    |
| **> 1.0**        | Potentially problematic      | ❌ Can invalidate LOO |


Because the pareto_k diagnostics show that fewer than 1% of observations are influential, 
meaning the LOO model comparison is stable and reliable.

In [16]:
loo2 = loo_diagnostics(idata_puma_pois, name="Poisson PUMA×DOW")
loo3 = loo_diagnostics(idata_puma_nta_pois, name="Poisson + NTA pooling")


/Users/mozilla/Library/Caches/pypoetry/virtualenvs/nyc-311-bayesian-noise-models-pweGKFeb-py3.12/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


[Poisson PUMA×DOW] frac pareto_k > 0.7: 0.007 | > 1.0: 0.002


/Users/mozilla/Library/Caches/pypoetry/virtualenvs/nyc-311-bayesian-noise-models-pweGKFeb-py3.12/lib/python3.12/site-packages/arviz/stats/stats.py:782: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.70 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


[Poisson + NTA pooling] frac pareto_k > 0.7: 0.009 | > 1.0: 0.003


### Export for visualization

Finally we follow similar steps as in section 2 for preparring to export and visualize in Kepler

In [17]:
# --- Posterior draws of lambda[puma, dow]
lam_post = idata_puma_nta_pois.posterior["lam"]  # dims: chain, draw, puma, dow

# Posterior mean
lam_mean = (
    lam_post
    .mean(dim=("chain", "draw"))
    .to_dataframe(name="lam_mean")
    .reset_index()
)

# 90% HDI
hdi_ds = az.hdi(lam_post, hdi_prob=0.9)
hdi_df = (
    hdi_ds["lam"]
    .to_dataframe(name="lam_hdi")
    .reset_index()
    .pivot_table(
        index=["puma", "dow"],
        columns="hdi",
        values="lam_hdi"
    )
    .reset_index()
    .rename(columns={"lower": "lam_low_90", "higher": "lam_high_90"})
)

# Combine
df_post = lam_mean.merge(hdi_df, on=["puma", "dow"], how="left")
df_post["lam_width_90"] = df_post["lam_high_90"] - df_post["lam_low_90"]

df_post.head()


,puma,dow,lam_mean,lam_high_90,lam_low_90,lam_width_90
0,4103,Monday,29.960130,31.371847,28.532858,2.838989
1,4103,Tuesday,28.630326,30.008695,27.199584,2.809111
2,4103,Wednesday,32.028599,33.432397,30.515913,2.916485
3,4103,Thursday,36.925421,38.473761,35.280891,3.192869
4,4103,Friday,47.493429,49.184505,45.822268,3.362237


In [18]:
# --- Map weekday → synthetic date
dow_to_date = {
    "Monday":    "2000-01-03",
    "Tuesday":   "2000-01-04",
    "Wednesday": "2000-01-05",
    "Thursday":  "2000-01-06",
    "Friday":    "2000-01-07",
    "Saturday":  "2000-01-08",
    "Sunday":    "2000-01-09",
}

df_post["date"] = pd.to_datetime(
    df_post["dow"].map(dow_to_date),
    errors="coerce"
).astype("datetime64[ns]")


In [19]:
gdf_kepler = kepler_typical_week_from_dow_complaint(
    df_post,
    puma_geojson_path="../data/raw/nyc/geographies/nyc_pumas_2020.geojson",
    out_path="../data/processed/kepler/03_01_nyc_puma_typical_week_model3_posterior.geojson",
)


✅ Kepler GeoJSON written to: ../data/processed/kepler/03_01_nyc_puma_typical_week_model3_posterior.geojson


### export for prediction

In [20]:

export_idata(idata_puma_nta_pois, "../data/processed/models/model2_puma_nta_idata.nc")

✅ Saved idata -> ../data/processed/models/model2_puma_nta_idata.nc


'../data/processed/models/model2_puma_nta_idata.nc'